**MODEL OPTIMIZATION*

Goal: Tune hyperparameters and compare multiple models

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import itertools
import time

In [14]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from src import config
PROCESSED_DATA_PATH = config.PROCESSED_DATA_PATH
MODEL_DIR = config.MODEL_PATH
BEST_PARAMS_PATH = config.PARAM_PATH
TFIFD_VEC_PATH = config.TFIFD_VEC_PATH
TAGS_VEC_PATH = config.TAGS_VEC_PATH
GENRES_VEC_PATH = config.GENRES_VEC_PATH
TFIFD_MATRIX_PATH = config.TFIFD_MATRIX_PATH
TAGS_MATRIX_PATH = config.TAGS_MATRIX_PATH
GENRES_MATRIX_PATH = config.GENRES_MATRIX_PATH

In [3]:
df = pd.read_csv(PROCESSED_DATA_PATH)

In [4]:
df['description'] = df['description'].fillna('')

In [5]:
df["description"] = df["description"].astype(str)
df["tags"] = df["tags"].astype(str)
df["genres"] = df["genres"].astype(str)

In [6]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

**Hyperparameter selection**

In [7]:
weights = {
    "desc_weight":  [0.6, 1.0],
    "tag_weight":   [0.3, 0.6],
    "genre_weight": [0.2, 0.5]
}

In [8]:
tfidf_params = {
    "max_features": [3000, 5000],
    "ngram_range": [(1,1), (1,2)],
    "min_df": [2, 5]
}

**Model Building**

In [9]:
def build_and_score(tfidf_dict, weight_dict):

    vec_desc = TfidfVectorizer(
        max_features=tfidf_dict["max_features"],
        ngram_range=tfidf_dict["ngram_range"],
        min_df=tfidf_dict["min_df"]
    )
    desc_train = vec_desc.fit_transform(train_df["description"])
    desc_test  = vec_desc.transform(test_df["description"])
    sim_desc   = cosine_similarity(desc_test, desc_train)

    vec_tags  = CountVectorizer(stop_words='english')
    tag_train = vec_tags.fit_transform(train_df["tags"])
    tag_test  = vec_tags.transform(test_df["tags"])
    sim_tags  = cosine_similarity(tag_test, tag_train)

    vec_genres  = CountVectorizer(stop_words='english')
    genre_train = vec_genres.fit_transform(train_df["genres"])
    genre_test  = vec_genres.transform(test_df["genres"])
    sim_genres  = cosine_similarity(genre_test, genre_train)

    w_d = weight_dict["desc_weight"]
    w_t = weight_dict["tag_weight"]
    w_g = weight_dict["genre_weight"]


    total_sim = (sim_desc * w_d) + (sim_tags * w_t) + (sim_genres * w_g)

    top_scores = total_sim.max(axis=1)
    return float(np.mean(top_scores))

In [10]:
results = []
start = time.time()
count = 0

In [ ]:
for tfidf_vals in itertools.product(*tfidf_params.values()):
    tfidf_dict = dict(zip(tfidf_params.keys(), tfidf_vals))

    for w_vals in itertools.product(*weights.values()):
        weight_dict = dict(zip(weights.keys(), w_vals))

        try:
            score = build_and_score(tfidf_dict, weight_dict)
            count += 1
            
            print(f"{count}) Score={score:.4f} | W_Desc:{weight_dict['desc_weight']}, W_Tag:{weight_dict['tag_weight']}, W_Genre:{weight_dict['genre_weight']}")
            
            results.append({
                "tfidf": tfidf_dict,
                "weights": weight_dict,
                "score": score
            })
        except Exception as e:
            print(f"Error: {e}")

end = time.time()
print(f"\nTOTAL TIME: {end - start:.2f} sec")

1) Score=0.5189 | W_Desc:0.6, W_Tag:0.3, W_Genre:0.2
2) Score=0.7940 | W_Desc:0.6, W_Tag:0.3, W_Genre:0.5
3) Score=0.7291 | W_Desc:0.6, W_Tag:0.6, W_Genre:0.2
4) Score=0.9936 | W_Desc:0.6, W_Tag:0.6, W_Genre:0.5
5) Score=0.6287 | W_Desc:1.0, W_Tag:0.3, W_Genre:0.2
6) Score=0.8911 | W_Desc:1.0, W_Tag:0.3, W_Genre:0.5
7) Score=0.8218 | W_Desc:1.0, W_Tag:0.6, W_Genre:0.2
8) Score=1.0780 | W_Desc:1.0, W_Tag:0.6, W_Genre:0.5
9) Score=0.5189 | W_Desc:0.6, W_Tag:0.3, W_Genre:0.2
10) Score=0.7941 | W_Desc:0.6, W_Tag:0.3, W_Genre:0.5
11) Score=0.7292 | W_Desc:0.6, W_Tag:0.6, W_Genre:0.2
12) Score=0.9937 | W_Desc:0.6, W_Tag:0.6, W_Genre:0.5
13) Score=0.6288 | W_Desc:1.0, W_Tag:0.3, W_Genre:0.2
14) Score=0.8912 | W_Desc:1.0, W_Tag:0.3, W_Genre:0.5
15) Score=0.8220 | W_Desc:1.0, W_Tag:0.6, W_Genre:0.2
16) Score=1.0782 | W_Desc:1.0, W_Tag:0.6, W_Genre:0.5
17) Score=0.5131 | W_Desc:0.6, W_Tag:0.3, W_Genre:0.2
18) Score=0.7889 | W_Desc:0.6, W_Tag:0.3, W_Genre:0.5
19) Score=0.7238 | W_Desc:0.6, W_Tag:

In [12]:
best = max(results, key=lambda x: x["score"])
print("BEST CONFIGURATION:")
print(best)

BEST CONFIGURATION:
{'tfidf': {'max_features': 3000, 'ngram_range': (1, 1), 'min_df': 5}, 'weights': {'desc_weight': 1.0, 'tag_weight': 0.6, 'genre_weight': 0.5}, 'score': 1.0781781381852837}


In [13]:
import pickle
import json

In [15]:
best_config = best  
with open(BEST_PARAMS_PATH, 'w') as f:
    json.dump(best_config, f, indent=4)
print("Done.")

Done.


In [16]:
t_params = best_config["tfidf"]

In [17]:
final_tfidf = TfidfVectorizer(
    max_features=t_params["max_features"],
    ngram_range=t_params["ngram_range"],
    min_df=t_params["min_df"]
)

In [18]:
tfidf_matrix = final_tfidf.fit_transform(df["description"])

In [19]:
final_tags_vec = CountVectorizer(stop_words='english')
tags_matrix = final_tags_vec.fit_transform(df["tags"])

In [20]:
final_genres_vec = CountVectorizer(stop_words='english')
genres_matrix = final_genres_vec.fit_transform(df["genres"])

In [21]:
with open(TFIFD_VEC_PATH, 'wb') as f:
    pickle.dump(final_tfidf, f)

In [22]:
with open(TAGS_VEC_PATH, 'wb') as f:
    pickle.dump(final_tags_vec, f)

In [23]:
with open(GENRES_VEC_PATH, 'wb') as f:
    pickle.dump(final_genres_vec, f)

In [24]:
with open(TFIFD_MATRIX_PATH, 'wb') as f:
    pickle.dump(tfidf_matrix, f)

In [25]:
with open(TAGS_MATRIX_PATH, 'wb') as f:
    pickle.dump(tags_matrix, f)

In [26]:
with open(GENRES_MATRIX_PATH, 'wb') as f:
    pickle.dump(genres_matrix, f)